![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Pipelines and Models

*Project 2 — Machine Learning: From Data to Predictions*

**Team: Zoé Doucet**
**Dataset: Covertype**

---

Steps 5 to 8 of the workflow: build the pipeline, compare models fairly, tune the best, and choose the threshold. Everything here uses the training set only.

> **Done when:** you have a comparison table of at least three model families on the same folds against the baseline, the best one or two tuned, and a final model chosen for reasons you can state.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the work, not whether you kept the scaffold.

---
## 0. Setup

In [27]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (f1_score, recall_score, precision_score, accuracy_score,
                             classification_report, ConfusionMatrixDisplay, make_scorer)
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load and split

The same loading and the same split as notebook 01. Put the code in `src/functions.py` so both notebooks use exactly the same rows.

In [ ]:
covertype= pd.read_csv("../data/covertype_clean.csv") # loading the data

In [19]:
# splitting the data using the split function
X, y, X_train, X_test, y_train, y_test, cv, folds = split( covertype.drop(columns= ["cover_type"]),
                                                    covertype["cover_type"],
                                                    0.25,
                                                    42,
                                                    5)

435759 training rows, 145253 test rows


---
## 2. The preprocessing pipeline

A `ColumnTransformer` with one branch per column type: impute, scale, encode. **Everything that learns from data goes inside**, so cross-validation refits it on every fold.

In [22]:
cat_features = ["soil_type", "wilderness_area"] # creating a lsit with the categorical features
num_features = X.drop(columns = cat_features).columns.tolist() # creating a list with the numerical features

preprocess = ColumnTransformer([
    ("num", 
      StandardScaler(), 
      num_features), # Scaling the numerical features
    ("cat", 
     OneHotEncoder(handle_unknown="ignore", sparse_output=False), 
     cat_features)])# transforming categrorical columns into binary columns


---
## 3. First model in the pipeline

One simple model at the end of the pipeline, scored with cross-validation. Is it better than the baseline, and by more than the spread across folds?

In [ ]:
log_reg_1 = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

In [30]:
cross_validation (pipe, X_train, y_train, folds)

Scores: {'f1': np.float64(0.533), 'recall': np.float64(0.512), 'accuracy': np.float64(0.725), 'precision': np.float64(0.604)} 
 Spread: {'f1': np.float64(0.005), 'recall': np.float64(0.005), 'accuracy': np.float64(0.002), 'precision': np.float64(0.008)}


({'f1': np.float64(0.533),
  'recall': np.float64(0.512),
  'accuracy': np.float64(0.725),
  'precision': np.float64(0.604)},
 {'f1': np.float64(0.005),
  'recall': np.float64(0.005),
  'accuracy': np.float64(0.002),
  'precision': np.float64(0.008)})

---
## 4. Features

New features added inside the pipeline. Keep a feature only if it improves the cross-validated score, and write down what each one changed.

In [ ]:
log_reg_2 = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

---
## 5. Compare models

At least three model families on **the same folds**, with the mean and the spread. A table, and a chart of it.

---
## 6. Tune

A `GridSearchCV` or `RandomizedSearchCV` on the best one or two, on the training set. Report the best parameters and the cross-validated score.

---
## 7. Threshold

Classification only. Cross-validated probabilities on the training set with `cross_val_predict`, then the threshold that fits your metric and the cost of each mistake. Regression: delete this section and say why in the README.

---
## 8. The final model

Which model, which parameters, which threshold, and why. If a simpler model is within the noise of the best one, say which you chose and why.